# FlowRec Yambda Filter Statistics

This notebook runs the same filter-statistics analysis as the Amazon notebook, but for the Yambda dataset. It uses FlowRec's Yambda loader, k-core filter, sequence split builder, and post-split cold-item filtering logic.

The default dataset is:

```python
YAMBDA_DATASETS = {
    "yambda_50m_listens": {
        "use_hf": True,
        "dataset_type": "sequential",
        "dataset_size": "50m",
        "event_file": "listens",
    },
}
```

The default configuration matches the requested global-time split and Yambda filters: `global_val_ratio=0.1`, `global_test_ratio=0.1`, `global_eval_target_mode="last_per_user"`, `global_history_policy="rolling"`, `min_played_ratio_pct=50`, `train_target_policy="last_n"`, `train_last_n=128`, `max_history_len=128`.


In [1]:
#!/usr/bin/env python3
"""
Analyze FlowRec/Yambda data statistics under different filtering settings.

This script uses FlowRec's Yambda data-processing modules instead of
reimplementing dataset parsing. It loads Yambda interactions via
``load_yambda_events_by_user`` or Hugging Face, applies FlowRec k-core filters,
uses FlowRec sequence splitting / post-split cold-item filtering logic, and then
saves CSV/JSON summaries plus plots.

Example from the repository root:

PYTHONPATH=. python scripts/analyze_flowrec_yambda_filter_stats.py \
  --datasets yambda_50m_listens \
  --split-strategy global_time \
  --global-val-ratio 0.1 \
  --global-test-ratio 0.1 \
  --global-history-policy rolling \
  --global-eval-target-mode last_per_user \
  --min-played-ratio-pct 50 \
  --train-all-prefixes false \
  --train-target-policy last_n \
  --train-last-n 128 \
  --max-history-len 128 \
  --min-user-values 5,10,20,50 \
  --min-item-values 5,10,20,50 \
  --batch-size 256 \
  --eval-batch-size 256 \
  --output-dir reports/yambda_filter_stats

For a quick smoke test, use ``--max-rows 100000``.
"""

from __future__ import annotations

import argparse
import gc
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path
from typing import Any, Mapping, Sequence

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from flowrec.data.datasets import SequenceExample
from flowrec.data.preprocessors import (
    apply_k_core_filter,
    count_interactions,
    deduplicate_consecutive_items,
    load_semantic_ids,
    sort_user_events,
)
from flowrec.data.post_split_filtering import (
    disabled_post_split_filter_stats,
    filter_splits_to_train_item_catalog,
)
from flowrec.data.sequence_builder import SequenceSplitConfig, SequenceSplits, build_sequence_splits
from flowrec.data.yambda import YambdaDataConfig, load_yambda_events_by_user


YAMBDA_DATASETS: dict[str, dict[str, Any]] = {
    "yambda_50m_listens": {
        "use_hf": True,
        "dataset_type": "sequential",
        "dataset_size": "50m",
        "event_file": "listens",
    },
}


def str_to_bool(value: str | bool) -> bool:
    if isinstance(value, bool):
        return value
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "y", "on"}:
        return True
    if normalized in {"0", "false", "no", "n", "off"}:
        return False
    raise argparse.ArgumentTypeError(f"Cannot parse boolean value: {value!r}")


def parse_int_list(value: str) -> list[int]:
    if value is None or value.strip() == "":
        return []
    return [int(part.strip()) for part in value.split(",") if part.strip()]


def parse_str_list(value: str) -> list[str]:
    if value is None or value.strip() == "" or value.strip().lower() == "all":
        return list(YAMBDA_DATASETS)
    return [part.strip() for part in value.split(",") if part.strip()]


def parse_optional_tuple(value: str | None) -> tuple[str, ...] | None:
    if value is None or value.strip() == "" or value.strip().lower() in {"none", "null"}:
        return None
    return tuple(part.strip() for part in value.split(",") if part.strip())


def ensure_dir(path: str | Path) -> Path:
    out = Path(path)
    out.mkdir(parents=True, exist_ok=True)
    return out


def quantile_stats(values: Sequence[int] | np.ndarray, prefix: str) -> dict[str, float | int]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return {
            f"{prefix}_count": 0,
            f"{prefix}_min": 0.0,
            f"{prefix}_mean": 0.0,
            f"{prefix}_std": 0.0,
            f"{prefix}_p25": 0.0,
            f"{prefix}_p50": 0.0,
            f"{prefix}_p75": 0.0,
            f"{prefix}_p90": 0.0,
            f"{prefix}_p95": 0.0,
            f"{prefix}_p99": 0.0,
            f"{prefix}_max": 0.0,
        }

    return {
        f"{prefix}_count": int(arr.size),
        f"{prefix}_min": float(np.min(arr)),
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr)),
        f"{prefix}_p25": float(np.quantile(arr, 0.25)),
        f"{prefix}_p50": float(np.quantile(arr, 0.50)),
        f"{prefix}_p75": float(np.quantile(arr, 0.75)),
        f"{prefix}_p90": float(np.quantile(arr, 0.90)),
        f"{prefix}_p95": float(np.quantile(arr, 0.95)),
        f"{prefix}_p99": float(np.quantile(arr, 0.99)),
        f"{prefix}_max": float(np.max(arr)),
    }


def make_yambda_cfg(
    args: argparse.Namespace,
    *,
    dataset_cfg: Mapping[str, Any] | None = None,
    min_user: int,
    min_item: int,
) -> YambdaDataConfig:
    dataset_cfg = dict(dataset_cfg or {})

    use_hf = bool(dataset_cfg.get("use_hf", args.use_hf))
    interactions_path = dataset_cfg.get("interactions_path", args.interactions_path)
    if interactions_path is not None:
        use_hf = False

    return YambdaDataConfig(
        interactions_path=interactions_path,
        semantic_id_path=args.semantic_id_path,
        use_hf=use_hf,
        hf_dataset_name=dataset_cfg.get("hf_dataset_name", args.hf_dataset_name),
        dataset_type=dataset_cfg.get("dataset_type", args.dataset_type),
        dataset_size=dataset_cfg.get("dataset_size", args.dataset_size),
        event_file=dataset_cfg.get("event_file", args.event_file),
        streaming=args.streaming,
        max_rows=args.max_rows,
        split_strategy=args.split_strategy,
        global_val_ratio=args.global_val_ratio,
        global_test_ratio=args.global_test_ratio,
        global_val_timestamp=args.global_val_timestamp,
        global_test_timestamp=args.global_test_timestamp,
        global_history_policy=args.global_history_policy,
        global_eval_target_mode=args.global_eval_target_mode,
        organic_filter=args.organic_filter,
        min_played_ratio_pct=args.min_played_ratio_pct,
        allowed_event_types=parse_optional_tuple(args.allowed_event_types),
        train_all_prefixes=args.train_all_prefixes,
        train_target_policy=args.train_target_policy,
        train_last_n=args.train_last_n,
        train_targets_per_user=args.train_targets_per_user,
        train_stride=args.train_stride,
        train_hybrid_recent_n=args.train_hybrid_recent_n,
        train_hybrid_random_n=args.train_hybrid_random_n,
        min_history_len=args.min_history_len,
        min_user_interactions=min_user,
        min_item_interactions=min_item,
        max_history_len=args.max_history_len,
        deduplicate_consecutive=args.deduplicate_consecutive,
        keep_raw_interactions=False,
        batch_size=args.batch_size,
        eval_batch_size=args.eval_batch_size,
        num_workers=0,
        pin_memory=False,
        persistent_workers=False,
        include_raw_in_batch=False,
        filter_cold_items_after_split=args.filter_cold_items_after_split,
        cold_item_train_source=args.cold_item_train_source,
        seed=args.seed,
    )


def load_base_user_events(cfg: YambdaDataConfig):
    """Load Yambda interactions using FlowRec's Yambda parser/normalizer."""
    allowed_items = None
    if cfg.semantic_id_path is not None and cfg.drop_items_missing_semantic_id:
        semantic_ids, _ = load_semantic_ids(
            cfg.semantic_id_path,
            offset_by_one=cfg.offset_semantic_ids_by_one,
        )
        allowed_items = set(semantic_ids)

    user_events = load_yambda_events_by_user(
        interactions_path=Path(cfg.interactions_path) if cfg.interactions_path else None,
        use_hf=cfg.use_hf,
        dataset_type=cfg.dataset_type,
        dataset_size=cfg.dataset_size,
        event_file=cfg.event_file,
        hf_dataset_name=cfg.hf_dataset_name,
        streaming=cfg.streaming,
        max_rows=cfg.max_rows,
        keep_raw=cfg.keep_raw_interactions,
        allowed_items=allowed_items,
        organic_filter=cfg.organic_filter,
        min_played_ratio_pct=cfg.min_played_ratio_pct,
        allowed_event_types=cfg.allowed_event_types,
        event_value_map=cfg.event_value_map,
        default_rating=cfg.default_rating,
    )

    if cfg.deduplicate_consecutive:
        user_events = deduplicate_consecutive_items(user_events)

    return sort_user_events(user_events)


def raw_distribution_stats_from_filtered(filtered_user_events) -> tuple[dict[str, int | float], dict[str, np.ndarray]]:
    """Compute raw interaction distributions after FlowRec filtering."""
    filtered_user_events = sort_user_events(filtered_user_events)
    chain_lengths = np.asarray([len(events) for events in filtered_user_events.values()], dtype=int)
    item_counter = Counter(
        event.raw_item_id for events in filtered_user_events.values() for event in events
    )
    item_counts = np.asarray(list(item_counter.values()), dtype=int)
    num_interactions = int(count_interactions(filtered_user_events))
    num_users = int(len(filtered_user_events))
    num_items = int(len(item_counter))

    stats: dict[str, int | float] = {
        "raw_num_users_after_kcore": num_users,
        "raw_num_items_after_kcore": num_items,
        "raw_num_interactions_after_kcore": num_interactions,
        "raw_density_after_kcore": (
            num_interactions / (num_users * num_items)
            if num_users > 0 and num_items > 0
            else 0.0
        ),
    }
    stats.update(quantile_stats(chain_lengths, "raw_user_chain_len"))
    stats.update(quantile_stats(item_counts, "raw_item_interaction_count"))
    return stats, {
        "raw_user_chain_len": chain_lengths,
        "raw_item_interaction_count": item_counts,
    }


def _sequence_split_config(cfg: YambdaDataConfig) -> SequenceSplitConfig:
    return SequenceSplitConfig(
        split_strategy=cfg.split_strategy,
        min_history_len=cfg.min_history_len,
        train_all_prefixes=cfg.train_all_prefixes,
        train_target_policy=cfg.train_target_policy,
        train_targets_per_user=cfg.train_targets_per_user,
        train_stride=cfg.train_stride,
        train_last_n=cfg.train_last_n,
        train_hybrid_recent_n=cfg.train_hybrid_recent_n,
        train_hybrid_random_n=cfg.train_hybrid_random_n,
        max_history_len=cfg.max_history_len,
        global_val_timestamp=cfg.global_val_timestamp,
        global_test_timestamp=cfg.global_test_timestamp,
        global_val_ratio=cfg.global_val_ratio,
        global_test_ratio=cfg.global_test_ratio,
        global_history_policy=cfg.global_history_policy,
        global_eval_target_mode=cfg.global_eval_target_mode,
        seed=cfg.seed,
    )


def build_splits_from_filtered_events(filtered_user_events, cfg: YambdaDataConfig) -> tuple[SequenceSplits, dict[str, Any]]:
    """Mirror YambdaDataModule.setup() from already-filtered user_events.

    This avoids reloading the 50M interaction dataset for every filter setting while
    still using FlowRec's split builder and post-split cold-item filtering.
    """
    user2id: dict[str, int] = {}
    id2user: list[str] = []
    item2id: dict[str, int] = {}
    id2item: list[str] = ["<PAD>"]

    def get_user_id(raw_user_id: str) -> int:
        if raw_user_id not in user2id:
            user2id[raw_user_id] = len(id2user)
            id2user.append(raw_user_id)
        return user2id[raw_user_id]

    def get_item_id(raw_item_id: str) -> int:
        if raw_item_id not in item2id:
            item2id[raw_item_id] = len(id2item)
            id2item.append(raw_item_id)
        return item2id[raw_item_id]

    split_cfg = _sequence_split_config(cfg)
    splits = build_sequence_splits(
        filtered_user_events,
        get_user_id=get_user_id,
        get_item_id=get_item_id,
        cfg=split_cfg,
    )

    if cfg.filter_cold_items_after_split:
        filter_result = filter_splits_to_train_item_catalog(
            splits,
            old_id2item=id2item,
            train_item_source=cfg.cold_item_train_source,
            min_history_len=cfg.min_history_len,
        )
        splits = filter_result.splits
        item2id = filter_result.item2id
        id2item = filter_result.id2item
        post_filter_stats = filter_result.stats
    else:
        post_filter_stats = disabled_post_split_filter_stats(
            splits=splits,
            num_items=len(id2item) - 1,
            train_item_source=cfg.cold_item_train_source,
            min_history_len=cfg.min_history_len,
        )

    metadata: dict[str, Any] = {
        "num_users": len(id2user),
        "num_items": len(id2item) - 1,
        "item_vocab_size": len(id2item),
        "post_filter": post_filter_stats.to_dict(),
    }
    return splits, metadata


def example_distribution_stats(examples: list[SequenceExample], prefix: str) -> tuple[dict, dict[str, np.ndarray]]:
    history_lengths = np.asarray([len(ex.history_item_ids) for ex in examples], dtype=int)
    target_positions = np.asarray([ex.target_position for ex in examples], dtype=int)
    users = {ex.user_id for ex in examples}
    target_items = [ex.target_item_id for ex in examples]
    target_item_counts = np.asarray(list(Counter(target_items).values()), dtype=int)

    observed_items_by_user: dict[int, set[int]] = defaultdict(set)
    for ex in examples:
        observed_items_by_user[ex.user_id].update(i for i in ex.history_item_ids if i > 0)
        if ex.target_item_id > 0:
            observed_items_by_user[ex.user_id].add(ex.target_item_id)
    visible_chain_lengths = np.asarray([len(v) for v in observed_items_by_user.values()], dtype=int)

    stats: dict[str, int | float] = {
        f"{prefix}_num_examples": int(len(examples)),
        f"{prefix}_num_users": int(len(users)),
        f"{prefix}_num_target_items": int(len(set(target_items))),
    }
    stats.update(quantile_stats(history_lengths, f"{prefix}_history_len"))
    stats.update(quantile_stats(target_positions, f"{prefix}_target_position"))
    stats.update(quantile_stats(target_item_counts, f"{prefix}_target_item_count"))
    stats.update(quantile_stats(visible_chain_lengths, f"{prefix}_visible_chain_len"))

    return stats, {
        f"{prefix}_history_len": history_lengths,
        f"{prefix}_target_position": target_positions,
        f"{prefix}_target_item_count": target_item_counts,
        f"{prefix}_visible_chain_len": visible_chain_lengths,
    }


def split_distribution_stats(filtered_user_events, cfg: YambdaDataConfig) -> tuple[dict, dict[str, np.ndarray]]:
    """Build FlowRec sequence splits and collect split-level statistics."""
    splits, metadata = build_splits_from_filtered_events(filtered_user_events, cfg)

    stats: dict[str, Any] = {
        "dm_dataset": "yambda",
        "dm_dataset_type": cfg.dataset_type,
        "dm_dataset_size": cfg.dataset_size,
        "dm_event_file": cfg.event_file,
        "dm_split_strategy": cfg.split_strategy,
        "dm_global_val_cutoff": splits.val_cutoff,
        "dm_global_test_cutoff": splits.test_cutoff,
        "dm_num_users": int(metadata["num_users"]),
        "dm_num_items": int(metadata["num_items"]),
        "dm_item_vocab_size": int(metadata["item_vocab_size"]),
        "dm_num_interactions": int(count_interactions(filtered_user_events)),
    }
    for key, value in metadata["post_filter"].items():
        if isinstance(value, (int, float, str, bool)) or value is None:
            stats[f"post_filter_{key}"] = value

    all_dists: dict[str, np.ndarray] = {}
    for split_name, examples in {
        "train": splits.train,
        "val": splits.val,
        "test": splits.test,
    }.items():
        split_stats, split_dists = example_distribution_stats(examples, split_name)
        stats.update(split_stats)
        all_dists.update(split_dists)

    return stats, all_dists


def metric_label(row: pd.Series) -> str:
    return f"u≥{int(row['min_user_interactions'])}, i≥{int(row['min_item_interactions'])}"


def plot_lines(
    summary: pd.DataFrame,
    metrics: list[str],
    *,
    title: str,
    ylabel: str,
    out_path: Path,
    log_y: bool = False,
) -> None:
    labels = summary["label"].tolist()
    x = np.arange(len(summary))
    fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(labels)), 6))
    for metric in metrics:
        if metric in summary.columns:
            ax.plot(x, summary[metric].astype(float), marker="o", label=metric)
    ax.set_title(title)
    ax.set_xlabel("Filter setting")
    ax.set_ylabel(ylabel)
    if log_y:
        ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def plot_quantiles(
    summary: pd.DataFrame,
    prefix: str,
    *,
    title: str,
    ylabel: str,
    out_path: Path,
    log_y: bool = False,
) -> None:
    metrics = [
        f"{prefix}_mean",
        f"{prefix}_p50",
        f"{prefix}_p75",
        f"{prefix}_p90",
        f"{prefix}_p95",
        f"{prefix}_p99",
    ]
    readable = {
        f"{prefix}_mean": "mean",
        f"{prefix}_p50": "p50",
        f"{prefix}_p75": "p75",
        f"{prefix}_p90": "p90",
        f"{prefix}_p95": "p95",
        f"{prefix}_p99": "p99",
    }
    labels = summary["label"].tolist()
    x = np.arange(len(summary))
    fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(labels)), 6))
    for metric in metrics:
        if metric in summary.columns:
            ax.plot(x, summary[metric].astype(float), marker="o", label=readable[metric])
    ax.set_title(title)
    ax.set_xlabel("Filter setting")
    ax.set_ylabel(ylabel)
    if log_y:
        ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def plot_stacked_examples(summary: pd.DataFrame, out_path: Path) -> None:
    labels = summary["label"].tolist()
    x = np.arange(len(summary))
    train = summary.get("train_num_examples", pd.Series([0] * len(summary))).astype(float)
    val = summary.get("val_num_examples", pd.Series([0] * len(summary))).astype(float)
    test = summary.get("test_num_examples", pd.Series([0] * len(summary))).astype(float)

    fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(labels)), 6))
    ax.bar(x, train, label="train")
    ax.bar(x, val, bottom=train, label="val")
    ax.bar(x, test, bottom=train + val, label="test")
    ax.set_title("Constructed sequence examples after split and post-filtering")
    ax.set_xlabel("Filter setting")
    ax.set_ylabel("Number of examples")
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=220)
    plt.close(fig)


def _format_heatmap_value(value: float) -> str:
    if not np.isfinite(value):
        return "nan"
    if abs(value) >= 100_000:
        return f"{value:.1e}"
    if abs(value) >= 1_000:
        return f"{value:,.0f}"
    if abs(value) >= 100:
        return f"{value:.0f}"
    if abs(value) >= 10:
        return f"{value:.1f}"
    return f"{value:.2f}"


def plot_heatmap(
    summary: pd.DataFrame,
    metric: str,
    *,
    title: str,
    out_path: Path,
    log_values: bool = False,
    annotation_fontsize: int = 13,
) -> None:
    """Heatmap with large white bold annotations and dark outline."""
    if metric not in summary.columns:
        return

    pivot = summary.pivot(
        index="min_user_interactions",
        columns="min_item_interactions",
        values=metric,
    ).sort_index().sort_index(axis=1)

    data = pivot.to_numpy(dtype=float)
    shown = np.log10(np.maximum(data, 1.0)) if log_values else data

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(shown, aspect="auto")

    ax.set_title(title, fontsize=14, pad=12)
    ax.set_xlabel("min_item_interactions", fontsize=12)
    ax.set_ylabel("min_user_interactions", fontsize=12)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns], fontsize=11)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([str(i) for i in pivot.index], fontsize=11)
    ax.set_xticks(np.arange(-0.5, len(pivot.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(pivot.index), 1), minor=True)
    ax.grid(which="minor", linewidth=0.7, alpha=0.35)
    ax.tick_params(which="minor", bottom=False, left=False)

    import matplotlib.patheffects as path_effects

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            annotation = ax.text(
                j,
                i,
                _format_heatmap_value(data[i, j]),
                ha="center",
                va="center",
                color="white",
                fontsize=annotation_fontsize,
                fontweight="bold",
            )
            annotation.set_path_effects([
                path_effects.Stroke(linewidth=2.2, foreground="black", alpha=0.55),
                path_effects.Normal(),
            ])

    cbar = fig.colorbar(im, ax=ax, label="log10(value)" if log_values else metric)
    cbar.ax.tick_params(labelsize=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=240, bbox_inches="tight")
    plt.close(fig)


def save_plots(summary: pd.DataFrame, out_dir: Path) -> None:
    plots_dir = out_dir / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    plot_lines(
        summary,
        ["raw_num_users_after_kcore", "raw_num_items_after_kcore", "raw_num_interactions_after_kcore"],
        title="Raw Yambda size after FlowRec k-core filtering",
        ylabel="Count",
        out_path=plots_dir / "raw_counts_after_kcore.png",
        log_y=True,
    )
    plot_lines(
        summary,
        ["dm_num_users", "dm_num_items", "dm_num_interactions"],
        title="Yambda catalog size after split and post-filtering",
        ylabel="Count",
        out_path=plots_dir / "datamodule_counts_after_split.png",
        log_y=True,
    )
    plot_stacked_examples(summary, plots_dir / "split_examples_stacked.png")

    plot_quantiles(
        summary,
        "raw_user_chain_len",
        title="Distribution of raw Yambda user chain lengths after filtering",
        ylabel="Interactions per user",
        out_path=plots_dir / "raw_user_chain_length_quantiles.png",
        log_y=True,
    )
    plot_quantiles(
        summary,
        "raw_item_interaction_count",
        title="Distribution of raw Yambda item interaction counts after filtering",
        ylabel="Interactions per item",
        out_path=plots_dir / "raw_item_interaction_count_quantiles.png",
        log_y=True,
    )

    for split in ("train", "val", "test"):
        plot_quantiles(
            summary,
            f"{split}_history_len",
            title=f"Distribution of {split} input history lengths",
            ylabel="History length in constructed examples",
            out_path=plots_dir / f"{split}_history_length_quantiles.png",
            log_y=False,
        )
        plot_quantiles(
            summary,
            f"{split}_target_item_count",
            title=f"Distribution of {split} target-item frequencies in examples",
            ylabel="Examples per target item",
            out_path=plots_dir / f"{split}_target_item_frequency_quantiles.png",
            log_y=True,
        )

    heatmaps = [
        ("raw_num_users_after_kcore", "Raw number of users after k-core", True),
        ("raw_num_items_after_kcore", "Raw number of items after k-core", True),
        ("raw_num_interactions_after_kcore", "Raw number of interactions after k-core", True),
        ("raw_user_chain_len_mean", "Mean raw user chain length", False),
        ("raw_user_chain_len_p95", "P95 raw user chain length", False),
        ("raw_item_interaction_count_mean", "Mean raw item interaction count", False),
        ("raw_item_interaction_count_p95", "P95 raw item interaction count", False),
        ("train_num_examples", "Number of train examples", True),
        ("val_num_examples", "Number of validation examples", True),
        ("test_num_examples", "Number of test examples", True),
        ("post_filter_num_items_removed", "Items removed by post-split cold filtering", True),
        ("post_filter_dropped_val_examples", "Validation examples dropped by cold filtering", True),
        ("post_filter_dropped_test_examples", "Test examples dropped by cold filtering", True),
    ]
    for metric, title, log_values in heatmaps:
        plot_heatmap(
            summary,
            metric,
            title=title,
            out_path=plots_dir / f"heatmap_{metric}.png",
            log_values=log_values,
        )


def analyze_dataset(
    args: argparse.Namespace,
    *,
    dataset_name: str,
    dataset_cfg: Mapping[str, Any],
    out_dir: Path,
) -> pd.DataFrame:
    out_dir = ensure_dir(out_dir)
    base_cfg = make_yambda_cfg(args, dataset_cfg=dataset_cfg, min_user=1, min_item=0)

    print("\n==============================")
    print(f"Dataset:        {dataset_name}")
    print(f"use_hf:         {base_cfg.use_hf}")
    print(f"dataset_type:   {base_cfg.dataset_type}")
    print(f"dataset_size:   {base_cfg.dataset_size}")
    print(f"event_file:     {base_cfg.event_file}")
    print(f"interactions:   {base_cfg.interactions_path}")
    print(f"min_played_pct: {base_cfg.min_played_ratio_pct}")
    print(f"max_rows:       {base_cfg.max_rows}")
    print(f"output_dir:     {out_dir}")
    print("==============================")

    min_user_values = parse_int_list(args.min_user_values)
    min_item_values = parse_int_list(args.min_item_values)
    if not min_user_values or not min_item_values:
        raise ValueError("Both --min-user-values and --min-item-values must be non-empty comma-separated lists.")

    print("Loading base interactions with FlowRec Yambda loader...")
    base_user_events = load_base_user_events(base_cfg)
    print(f"Loaded {len(base_user_events):,} users before k-core grid evaluation.")
    print(f"Loaded {count_interactions(base_user_events):,} interactions after event filters.")

    rows: list[dict[str, Any]] = []
    for min_user in min_user_values:
        for min_item in min_item_values:
            print(f"\n=== {dataset_name}: min_user_interactions={min_user}, min_item_interactions={min_item} ===")
            cfg = make_yambda_cfg(args, dataset_cfg=dataset_cfg, min_user=min_user, min_item=min_item)

            filtered_events = apply_k_core_filter(
                base_user_events,
                min_user_interactions=min_user,
                min_item_interactions=min_item,
            )
            filtered_events = sort_user_events(filtered_events)

            raw_stats, _ = raw_distribution_stats_from_filtered(filtered_events)
            split_stats, _ = split_distribution_stats(filtered_events, cfg)

            row: dict[str, Any] = {
                "dataset": dataset_name,
                "min_user_interactions": min_user,
                "min_item_interactions": min_item,
                "min_played_ratio_pct": cfg.min_played_ratio_pct,
                "split_strategy": cfg.split_strategy,
                "global_val_ratio": cfg.global_val_ratio,
                "global_test_ratio": cfg.global_test_ratio,
                "global_history_policy": cfg.global_history_policy,
                "global_eval_target_mode": cfg.global_eval_target_mode,
                "train_target_policy": cfg.train_target_policy,
                "train_last_n": cfg.train_last_n,
                "max_history_len": cfg.max_history_len,
                **raw_stats,
                **split_stats,
            }
            rows.append(row)
            print(
                "raw users/items/interactions="
                f"{raw_stats['raw_num_users_after_kcore']:,}/"
                f"{raw_stats['raw_num_items_after_kcore']:,}/"
                f"{raw_stats['raw_num_interactions_after_kcore']:,}; "
                "examples train/val/test="
                f"{split_stats.get('train_num_examples', 0):,}/"
                f"{split_stats.get('val_num_examples', 0):,}/"
                f"{split_stats.get('test_num_examples', 0):,}"
            )
            gc.collect()

    summary = pd.DataFrame(rows)
    summary["label"] = summary.apply(metric_label, axis=1)
    summary = summary.sort_values(["min_user_interactions", "min_item_interactions"]).reset_index(drop=True)

    summary_path = out_dir / f"flowrec_yambda_filter_statistics_{dataset_name}.csv"
    summary.to_csv(summary_path, index=False)
    summary.to_json(out_dir / f"flowrec_yambda_filter_statistics_{dataset_name}.json", orient="records", indent=2)
    save_plots(summary, out_dir)

    print(f"\nSaved dataset summary: {summary_path}")
    print(f"Saved dataset plots:   {out_dir / 'plots'}")
    return summary


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Analyze FlowRec/Yambda data statistics across filter settings.")

    p.add_argument("--datasets", default="yambda_50m_listens", help="Comma-separated YAMBDA_DATASETS names, or 'all'.")
    p.add_argument("--interactions-path", default=None, help="Optional local file path. If set, use_hf is disabled.")
    p.add_argument("--semantic-id-path", default=None)
    p.add_argument("--use-hf", type=str_to_bool, default=True)
    p.add_argument("--hf-dataset-name", default="yandex/yambda")
    p.add_argument("--dataset-type", default="sequential", choices=["flat", "sequential"])
    p.add_argument("--dataset-size", default="50m", choices=["50m", "500m", "5b"])
    p.add_argument("--event-file", default="listens", choices=["listens", "likes", "dislikes", "unlikes", "undislikes", "multi_event"])
    p.add_argument("--streaming", type=str_to_bool, default=False)
    p.add_argument("--max-rows", type=int, default=None, help="Optional row cap for debugging/smoke tests.")

    p.add_argument("--split-strategy", default="global_time", choices=["leave_one_out", "global_time"])
    p.add_argument("--global-val-ratio", type=float, default=0.1)
    p.add_argument("--global-test-ratio", type=float, default=0.1)
    p.add_argument("--global-val-timestamp", type=int, default=None)
    p.add_argument("--global-test-timestamp", type=int, default=None)
    p.add_argument("--global-history-policy", default="rolling", choices=["rolling", "window_start"])
    p.add_argument("--global-eval-target-mode", default="last_per_user", choices=["all", "last_per_user"])

    p.add_argument("--min-played-ratio-pct", type=float, default=50.0)
    p.add_argument("--organic-filter", default="any", choices=["any", "organic", "recommendation"])
    p.add_argument("--allowed-event-types", default=None, help="Comma-separated event types, or empty/None.")

    p.add_argument("--train-all-prefixes", type=str_to_bool, default=False)
    p.add_argument("--train-target-policy", default="last_n", choices=["all", "last", "last_n", "stride", "sampled", "hybrid"])
    p.add_argument("--train-last-n", type=int, default=128)
    p.add_argument("--train-targets-per-user", type=int, default=256)
    p.add_argument("--train-stride", type=int, default=10)
    p.add_argument("--train-hybrid-recent-n", type=int, default=256)
    p.add_argument("--train-hybrid-random-n", type=int, default=256)
    p.add_argument("--min-history-len", type=int, default=1)
    p.add_argument("--max-history-len", type=int, default=128)

    p.add_argument("--min-user-values", default="5,10,20,50")
    p.add_argument("--min-item-values", default="5,10,20,50")

    p.add_argument("--deduplicate-consecutive", action="store_true")
    p.add_argument("--filter-cold-items-after-split", type=str_to_bool, default=True)
    p.add_argument("--cold-item-train-source", default="observed", choices=["observed", "targets"])

    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--eval-batch-size", type=int, default=256)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--output-dir", required=True)

    return p.parse_args()


def main() -> None:
    args = parse_args()
    output_root = ensure_dir(args.output_dir)

    if args.interactions_path:
        dataset_cfg = {
            "use_hf": False,
            "interactions_path": args.interactions_path,
            "dataset_type": args.dataset_type,
            "dataset_size": args.dataset_size,
            "event_file": args.event_file,
        }
        summary = analyze_dataset(
            args,
            dataset_name="custom_yambda",
            dataset_cfg=dataset_cfg,
            out_dir=output_root,
        )
        print(f"\nSaved summary: {output_root / 'flowrec_yambda_filter_statistics_custom_yambda.csv'}")
        return

    selected = parse_str_list(args.datasets)
    unknown = [name for name in selected if name not in YAMBDA_DATASETS]
    if unknown:
        raise ValueError(f"Unknown Yambda dataset names: {unknown}. Available: {list(YAMBDA_DATASETS)}")

    all_summaries: list[pd.DataFrame] = []
    for dataset_name in selected:
        dataset_summary = analyze_dataset(
            args,
            dataset_name=dataset_name,
            dataset_cfg=YAMBDA_DATASETS[dataset_name],
            out_dir=output_root / dataset_name,
        )
        all_summaries.append(dataset_summary)

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
    combined_path = output_root / "flowrec_yambda_filter_statistics_all_datasets.csv"
    combined.to_csv(combined_path, index=False)
    combined.to_json(output_root / "flowrec_yambda_filter_statistics_all_datasets.json", orient="records", indent=2)
    print(f"\nSaved combined summary: {combined_path}")



In [2]:

# --- Notebook configuration ---
# Run this notebook from the repository root so that `flowrec` is importable.
# For a quick smoke test, set MAX_ROWS to a smaller value such as 100_000.

from argparse import Namespace
from pathlib import Path
import pandas as pd

seed = 42
OUTPUT_DIR = "reports/yambda_filter_stats"
MAX_ROWS = None  # e.g. 100_000 for a quick test; None for full Yambda 50m listens

YAMBDA_DATASETS = {
    "yambda_50m_listens": {
        "use_hf": True,
        "dataset_type": "sequential",
        "dataset_size": "50m",
        "event_file": "listens",
    },
}

selected_datasets = ["yambda_50m_listens"]

args = Namespace(
    datasets=",".join(selected_datasets),
    interactions_path=None,
    semantic_id_path=None,
    use_hf=True,
    hf_dataset_name="yandex/yambda",
    dataset_type="sequential",
    dataset_size="50m",
    event_file="listens",
    streaming=False,
    max_rows=MAX_ROWS,

    # Global time split.
    split_strategy="global_time",
    global_val_ratio=0.1,
    global_test_ratio=0.1,
    global_val_timestamp=None,
    global_test_timestamp=None,
    global_history_policy="rolling",
    global_eval_target_mode="last_per_user",

    # Yambda event filtering.
    min_played_ratio_pct=50.0,
    organic_filter="any",
    allowed_event_types=None,

    # Training-example construction.
    train_all_prefixes=False,
    train_target_policy="last_n",
    train_last_n=128,
    train_targets_per_user=256,
    train_stride=10,
    train_hybrid_recent_n=256,
    train_hybrid_random_n=256,
    min_history_len=1,
    max_history_len=128,

    # Filter grid. Change these to control the heatmap axes.
    min_user_values="5,10,20,50",
    min_item_values="5,10,20,50",

    deduplicate_consecutive=False,
    filter_cold_items_after_split=True,
    cold_item_train_source="observed",

    batch_size=256,
    eval_batch_size=256,
    seed=seed,
    output_dir=OUTPUT_DIR,
)


In [3]:

# --- Run the full analysis ---
output_root = ensure_dir(args.output_dir)

all_summaries = []
for dataset_name in selected_datasets:
    if dataset_name not in YAMBDA_DATASETS:
        raise ValueError(f"Unknown Yambda dataset: {dataset_name}. Available: {list(YAMBDA_DATASETS)}")

    dataset_summary = analyze_dataset(
        args,
        dataset_name=dataset_name,
        dataset_cfg=YAMBDA_DATASETS[dataset_name],
        out_dir=output_root / dataset_name,
    )
    all_summaries.append(dataset_summary)

combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
combined_path = output_root / "flowrec_yambda_filter_statistics_all_datasets.csv"
combined.to_csv(combined_path, index=False)
combined.to_json(output_root / "flowrec_yambda_filter_statistics_all_datasets.json", orient="records", indent=2)
print(f"Saved combined summary: {combined_path}")
combined



Dataset:        yambda_50m_listens
use_hf:         True
dataset_type:   sequential
dataset_size:   50m
event_file:     listens
interactions:   None
min_played_pct: 50.0
max_rows:       None
output_dir:     reports/yambda_filter_stats/yambda_50m_listens
Loading base interactions with FlowRec Yambda loader...


/opt/anaconda3/envs/synth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 9,209 users before k-core grid evaluation.
Loaded 29,439,278 interactions after event filters.

=== yambda_50m_listens: min_user_interactions=5, min_item_interactions=5 ===
raw users/items/interactions=9,112/276,302/28,819,513; examples train/val/test=997,766/7,280/7,626

=== yambda_50m_listens: min_user_interactions=5, min_item_interactions=10 ===
raw users/items/interactions=9,106/194,311/28,277,160; examples train/val/test=995,502/7,400/7,776

=== yambda_50m_listens: min_user_interactions=5, min_item_interactions=20 ===
raw users/items/interactions=9,096/132,952/27,437,061; examples train/val/test=992,527/7,507/7,903

=== yambda_50m_listens: min_user_interactions=5, min_item_interactions=50 ===
raw users/items/interactions=9,078/76,118/25,667,008; examples train/val/test=986,936/7,590/8,017

=== yambda_50m_listens: min_user_interactions=10, min_item_interactions=5 ===
raw users/items/interactions=9,046/276,287/28,818,993; examples train/val/test=997,530/7,273/7,593

=== yambd

,dataset,min_user_interactions,min_item_interactions,min_played_ratio_pct,split_strategy,global_val_ratio,global_test_ratio,global_history_policy,global_eval_target_mode,train_target_policy,...,test_visible_chain_len_mean,test_visible_chain_len_std,test_visible_chain_len_p25,test_visible_chain_len_p50,test_visible_chain_len_p75,test_visible_chain_len_p90,test_visible_chain_len_p95,test_visible_chain_len_p99,test_visible_chain_len_max,label
0,yambda_50m_listens,5,5,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,87.247836,29.656424,68.0,94.0,111.0,120.0,124.0,128.0,129.0,"u≥5, i≥5"
1,yambda_50m_listens,5,10,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,88.101466,30.018618,69.0,95.0,113.0,122.0,125.0,128.0,129.0,"u≥5, i≥10"
2,yambda_50m_listens,5,20,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,89.009743,30.512208,69.0,96.0,114.0,123.0,126.0,128.0,129.0,"u≥5, i≥20"
3,yambda_50m_listens,5,50,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,89.337657,31.095950,69.0,97.0,115.0,124.0,127.0,129.0,129.0,"u≥5, i≥50"
4,yambda_50m_listens,10,5,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,87.598051,29.235550,69.0,94.0,111.0,120.0,124.0,128.0,129.0,"u≥10, i≥5"
5,yambda_50m_listens,10,10,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,88.480233,29.564321,69.0,95.0,113.0,122.0,125.0,128.0,129.0,"u≥10, i≥10"
6,yambda_50m_listens,10,20,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,89.398296,30.047469,70.0,97.0,114.0,123.0,126.0,128.0,129.0,"u≥10, i≥20"
7,yambda_50m_listens,10,50,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,89.786305,30.569262,70.0,97.0,115.0,124.0,127.0,129.0,129.0,"u≥10, i≥50"
8,yambda_50m_listens,20,5,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,88.120939,28.637295,69.0,94.0,112.0,121.0,124.0,128.0,129.0,"u≥20, i≥5"
9,yambda_50m_listens,20,10,50.0,global_time,0.1,0.1,rolling,last_per_user,last_n,...,89.010537,28.958146,70.0,95.0,113.0,122.0,125.0,128.0,129.0,"u≥20, i≥10"


In [4]:

# --- Inspect the most important columns ---
important_cols = [
    "dataset",
    "min_user_interactions",
    "min_item_interactions",
    "raw_num_users_after_kcore",
    "raw_num_items_after_kcore",
    "raw_num_interactions_after_kcore",
    "raw_user_chain_len_mean",
    "raw_user_chain_len_p95",
    "raw_item_interaction_count_mean",
    "raw_item_interaction_count_p95",
    "train_num_examples",
    "val_num_examples",
    "test_num_examples",
    "post_filter_num_items_removed",
    "post_filter_dropped_val_examples",
    "post_filter_dropped_test_examples",
]
combined[[c for c in important_cols if c in combined.columns]]


,dataset,min_user_interactions,min_item_interactions,raw_num_users_after_kcore,raw_num_items_after_kcore,raw_num_interactions_after_kcore,raw_user_chain_len_mean,raw_user_chain_len_p95,raw_item_interaction_count_mean,raw_item_interaction_count_p95,train_num_examples,val_num_examples,test_num_examples,post_filter_num_items_removed,post_filter_dropped_val_examples,post_filter_dropped_test_examples
0,yambda_50m_listens,5,5,9112,276302,28819513,3162.808714,11011.60,104.304395,412.00,997766,7280,7626,106464,823,1252
1,yambda_50m_listens,5,10,9106,194311,28277160,3105.332748,10766.00,145.525266,571.00,995502,7400,7776,49518,691,1083
2,yambda_50m_listens,5,20,9096,132952,27437061,3016.387533,10486.50,206.368170,793.00,992527,7507,7903,18556,570,933
3,yambda_50m_listens,5,50,9078,76118,25667008,2827.385768,9769.75,337.200242,1231.15,986936,7590,8017,3456,460,783
4,yambda_50m_listens,10,5,9046,276287,28818993,3185.827216,11038.75,104.308176,412.00,997530,7273,7593,106483,820,1244
5,yambda_50m_listens,10,10,9033,194307,28276618,3130.368427,10865.40,145.525473,571.00,995235,7392,7740,49536,687,1074
6,yambda_50m_listens,10,20,9021,132950,27436502,3041.403614,10507.00,206.367070,793.00,992261,7499,7866,18567,566,924
7,yambda_50m_listens,10,50,8998,76116,25666364,2852.452100,9787.75,337.200641,1231.25,986667,7575,7974,3457,459,776
8,yambda_50m_listens,20,5,8933,276266,28817338,3225.941789,11058.60,104.310114,412.00,996628,7254,7541,106522,806,1227
9,yambda_50m_listens,20,10,8927,194299,28275065,3167.364736,10906.80,145.523472,571.00,994375,7364,7687,49569,684,1064
